# Human Activity Recognition - Training Model

## Obiettivo
Creare e valutare modelli CNN per riconoscimento attività umane utilizzando:
- **1 sensore** posizionato al **braccio (wrist)**
- **Sensori**: Accelerometro, Giroscopio, Magnetometro
- **Frequenze**: 25Hz e 13Hz

## Configurazioni da testare
| Config | Sensori | Frequenza | Input Shape | Canali |
|--------|---------|-----------|-------------|--------|
| A | Acc | 25Hz | (64, 3) | 3 |
| B | Acc | 13Hz | (32, 3) | 3 |
| C | Acc + Gyro | 25Hz | (64, 6) | 6 |
| D | Acc + Gyro | 13Hz | (32, 6) | 6 |
| E | Acc + Gyro + Mag | 25Hz | (64, 9) | 9 |
| F | Acc + Gyro + Mag | 13Hz | (32, 9) | 9 |

## Attività da riconoscere
1. Walking (Camminare)
2. Running (Correre)
3. Standing (In piedi)
4. Sitting (Seduto)
5. Upstairs (Salire scale)
6. Downstairs (Scendere scale)

---
## 1. Setup e Import Librerie

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
import time
import os

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

print(f"TensorFlow version: {tf.__version__}")
print(f"Numpy version: {np.__version__}")

# Seed per riproducibilità
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Configurazione
ACTIVITIES = ['Walking', 'Running', 'Standing', 'Sitting', 'Upstairs', 'Downstairs']
NUM_CLASSES = len(ACTIVITIES)
ACTIVITY_MAP = {activity: i for i, activity in enumerate(ACTIVITIES)}

# Parametri sensori
ACC_RANGE = 16  # ±16g
GYRO_RANGE = 2000  # ±2000°/s
MAG_RANGE = 4900  # ±4900µT

# Parametri finestre
FREQ_HIGH = 25  # Hz
FREQ_LOW = 13   # Hz
WINDOW_SIZE_HIGH = 64  # samples @ 25Hz = 2.56s
WINDOW_SIZE_LOW = 32   # samples @ 13Hz = ~2.46s
OVERLAP = 0.5  # 50% overlap

# Dataset
SAMPLES_PER_ACTIVITY = 5000  # finestre per attività

print(f"Attività: {ACTIVITIES}")
print(f"Frequenze: {FREQ_HIGH}Hz ({WINDOW_SIZE_HIGH} samples), {FREQ_LOW}Hz ({WINDOW_SIZE_LOW} samples)")
print(f"Overlap: {OVERLAP*100}%")
print(f"Samples per activity: {SAMPLES_PER_ACTIVITY}")

---
## 2. Generazione Dati Sintetici

In [ ]:
def generate_sensor_signal(activity, sensor_type, duration, freq):
    """
    Genera segnale sintetico per un sensore specifico.
    
    Parameters:
    - activity: nome attività
    - sensor_type: 'acc', 'gyro', 'mag'
    - duration: durata in secondi
    - freq: frequenza di campionamento (Hz)
    
    Returns:
    - signal: array (n_samples, 3) per x,y,z
    """
    n_samples = int(duration * freq)
    t = np.linspace(0, duration, n_samples)
    
    # Parametri realistici per attività (valori medi e frequenze dominanti)
    params = {
        'Walking': {
            'acc': {'mean': [0.2, 9.8, 0.3], 'amp': [2.0, 3.0, 1.5], 'freq': [2, 2, 2]},
            'gyro': {'mean': [0, 0, 0], 'amp': [50, 30, 80], 'freq': [2, 2, 2]},
            'mag': {'mean': [20, -10, 40], 'amp': [5, 5, 5], 'freq': [1, 1, 1]}
        },
        'Running': {
            'acc': {'mean': [0.5, 9.8, 0.5], 'amp': [5.0, 7.0, 4.0], 'freq': [3, 3, 3]},
            'gyro': {'mean': [0, 0, 0], 'amp': [150, 100, 200], 'freq': [3, 3, 3]},
            'mag': {'mean': [20, -10, 40], 'amp': [8, 8, 8], 'freq': [1.5, 1.5, 1.5]}
        },
        'Standing': {
            'acc': {'mean': [0, 9.8, 0], 'amp': [0.1, 0.1, 0.1], 'freq': [0.5, 0.5, 0.5]},
            'gyro': {'mean': [0, 0, 0], 'amp': [5, 5, 5], 'freq': [0.3, 0.3, 0.3]},
            'mag': {'mean': [20, -10, 40], 'amp': [2, 2, 2], 'freq': [0.2, 0.2, 0.2]}
        },
        'Sitting': {
            'acc': {'mean': [0, 9.8, 0], 'amp': [0.05, 0.05, 0.05], 'freq': [0.3, 0.3, 0.3]},
            'gyro': {'mean': [0, 0, 0], 'amp': [2, 2, 2], 'freq': [0.2, 0.2, 0.2]},
            'mag': {'mean': [20, -10, 40], 'amp': [1, 1, 1], 'freq': [0.1, 0.1, 0.1]}
        },
        'Upstairs': {
            'acc': {'mean': [0.3, 10.5, 0.2], 'amp': [3.0, 4.5, 2.0], 'freq': [1.5, 1.5, 1.5]},
            'gyro': {'mean': [0, 0, 0], 'amp': [100, 80, 120], 'freq': [1.5, 1.5, 1.5]},
            'mag': {'mean': [20, -10, 40], 'amp': [6, 6, 6], 'freq': [1, 1, 1]}
        },
        'Downstairs': {
            'acc': {'mean': [0.2, 9.2, 0.3], 'amp': [3.5, 5.0, 2.5], 'freq': [1.5, 1.5, 1.5]},
            'gyro': {'mean': [0, 0, 0], 'amp': [120, 90, 140], 'freq': [1.5, 1.5, 1.5]},
            'mag': {'mean': [20, -10, 40], 'amp': [7, 7, 7], 'freq': [1, 1, 1]}
        }
    }
    
    p = params[activity][sensor_type]
    signal_data = np.zeros((n_samples, 3))
    
    for axis in range(3):
        # Componente principale (sinusoide)
        main_component = p['mean'][axis] + p['amp'][axis] * np.sin(2 * np.pi * p['freq'][axis] * t)
        
        # Componenti armoniche
        harmonic1 = (p['amp'][axis] * 0.3) * np.sin(2 * np.pi * p['freq'][axis] * 2 * t + np.pi/4)
        harmonic2 = (p['amp'][axis] * 0.15) * np.sin(2 * np.pi * p['freq'][axis] * 3 * t + np.pi/3)
        
        # Rumore gaussiano
        noise = np.random.normal(0, p['amp'][axis] * 0.1, n_samples)
        
        signal_data[:, axis] = main_component + harmonic1 + harmonic2 + noise
    
    return signal_data

In [ ]:
def create_windows(data, window_size, overlap=0.5):
    """
    Crea finestre sliding da segnale continuo.
    
    Parameters:
    - data: array (n_samples, n_channels)
    - window_size: dimensione finestra
    - overlap: percentuale overlap (0-1)
    
    Returns:
    - windows: array (n_windows, window_size, n_channels)
    """
    step = int(window_size * (1 - overlap))
    n_windows = (len(data) - window_size) // step + 1
    
    windows = []
    for i in range(n_windows):
        start = i * step
        end = start + window_size
        if end <= len(data):
            windows.append(data[start:end])
    
    return np.array(windows)

In [ ]:
def generate_dataset(freq, window_size, samples_per_activity=5000):
    """
    Genera dataset completo per una frequenza specifica.
    
    Returns:
    - X_acc: accelerometro (n_samples, window_size, 3)
    - X_gyro: giroscopio (n_samples, window_size, 3)
    - X_mag: magnetometro (n_samples, window_size, 3)
    - y: labels (n_samples,)
    """
    X_acc_list, X_gyro_list, X_mag_list, y_list = [], [], [], []
    
    for activity in ACTIVITIES:
        print(f"Generando dati per {activity} @ {freq}Hz...")
        
        # Durata necessaria per ottenere samples_per_activity finestre
        step = int(window_size * (1 - OVERLAP))
        n_samples_needed = samples_per_activity * step + window_size
        duration = n_samples_needed / freq
        
        # Genera segnali continui
        acc_signal = generate_sensor_signal(activity, 'acc', duration, freq)
        gyro_signal = generate_sensor_signal(activity, 'gyro', duration, freq)
        mag_signal = generate_sensor_signal(activity, 'mag', duration, freq)
        
        # Crea finestre
        acc_windows = create_windows(acc_signal, window_size, OVERLAP)
        gyro_windows = create_windows(gyro_signal, window_size, OVERLAP)
        mag_windows = create_windows(mag_signal, window_size, OVERLAP)
        
        # Prendi solo samples_per_activity finestre
        n_windows = min(samples_per_activity, len(acc_windows))
        
        X_acc_list.append(acc_windows[:n_windows])
        X_gyro_list.append(gyro_windows[:n_windows])
        X_mag_list.append(mag_windows[:n_windows])
        y_list.append(np.full(n_windows, ACTIVITY_MAP[activity]))
    
    X_acc = np.vstack(X_acc_list)
    X_gyro = np.vstack(X_gyro_list)
    X_mag = np.vstack(X_mag_list)
    y = np.concatenate(y_list)
    
    print(f"\nDataset generato:")
    print(f"  X_acc shape: {X_acc.shape}")
    print(f"  X_gyro shape: {X_gyro.shape}")
    print(f"  X_mag shape: {X_mag.shape}")
    print(f"  y shape: {y.shape}")
    print(f"  Distribuzione classi: {np.bincount(y)}")
    
    return X_acc, X_gyro, X_mag, y

In [ ]:
# Genera dataset per entrambe le frequenze
print("=" * 60)
print("GENERAZIONE DATASET 25Hz")
print("=" * 60)
X_acc_25, X_gyro_25, X_mag_25, y_25 = generate_dataset(FREQ_HIGH, WINDOW_SIZE_HIGH, SAMPLES_PER_ACTIVITY)

print("\n" + "=" * 60)
print("GENERAZIONE DATASET 13Hz")
print("=" * 60)
X_acc_13, X_gyro_13, X_mag_13, y_13 = generate_dataset(FREQ_LOW, WINDOW_SIZE_LOW, SAMPLES_PER_ACTIVITY)

---
## 3. Visualizzazione Dati Sintetici

In [ ]:
# Visualizza esempi di segnali per ogni attività
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(15, 12))
fig.suptitle('Esempi Segnali Sintetici - Accelerometro @ 25Hz', fontsize=16)

for i, activity in enumerate(ACTIVITIES):
    # Trova prima finestra di questa attività
    idx = np.where(y_25 == i)[0][0]
    sample = X_acc_25[idx]
    
    axes[i, 0].plot(sample[:, 0])
    axes[i, 0].set_ylabel(activity)
    axes[i, 0].set_title('Acc X' if i == 0 else '')
    axes[i, 0].grid(True, alpha=0.3)
    
    axes[i, 1].plot(sample[:, 1])
    axes[i, 1].set_title('Acc Y' if i == 0 else '')
    axes[i, 1].grid(True, alpha=0.3)
    
    axes[i, 2].plot(sample[:, 2])
    axes[i, 2].set_title('Acc Z' if i == 0 else '')
    axes[i, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Preprocessing e Normalizzazione

In [ ]:
def normalize_data(X_train, X_test):
    """
    Normalizza dati usando media e std del training set.
    """
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)
    
    X_train_norm = (X_train - mean) / (std + 1e-8)
    X_test_norm = (X_test - mean) / (std + 1e-8)
    
    return X_train_norm, X_test_norm, mean, std

In [ ]:
def prepare_data_for_config(X_acc, X_gyro, X_mag, y, sensors=['acc', 'gyro', 'mag']):
    """
    Prepara dati concatenando i sensori richiesti.
    
    Parameters:
    - sensors: lista di sensori da includere ['acc'], ['acc', 'gyro'], ['acc', 'gyro', 'mag']
    
    Returns:
    - X_combined: dati concatenati
    """
    data_list = []
    if 'acc' in sensors:
        data_list.append(X_acc)
    if 'gyro' in sensors:
        data_list.append(X_gyro)
    if 'mag' in sensors:
        data_list.append(X_mag)
    
    X_combined = np.concatenate(data_list, axis=2)
    return X_combined

---
## 5. Definizione Architettura CNN

In [ ]:
def create_cnn_model(input_shape, num_classes=6):
    """
    Crea modello CNN per HAR.
    
    Parameters:
    - input_shape: (timesteps, channels)
    - num_classes: numero di attività
    
    Returns:
    - model: modello Keras compilato
    """
    model = models.Sequential([
        # Primo blocco Conv
        layers.Conv1D(64, kernel_size=5, activation='relu', input_shape=input_shape, padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.2),
        
        # Secondo blocco Conv
        layers.Conv1D(128, kernel_size=5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.3),
        
        # Terzo blocco Conv
        layers.Conv1D(256, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        
        # Dense layers
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

---
## 6. Training per Tutte le Configurazioni

In [ ]:
# Configurazioni da testare
configs = [
    {'name': 'A_Acc_25Hz', 'sensors': ['acc'], 'freq': 25, 'X_acc': X_acc_25, 'X_gyro': X_gyro_25, 'X_mag': X_mag_25, 'y': y_25},
    {'name': 'B_Acc_13Hz', 'sensors': ['acc'], 'freq': 13, 'X_acc': X_acc_13, 'X_gyro': X_gyro_13, 'X_mag': X_mag_13, 'y': y_13},
    {'name': 'C_AccGyro_25Hz', 'sensors': ['acc', 'gyro'], 'freq': 25, 'X_acc': X_acc_25, 'X_gyro': X_gyro_25, 'X_mag': X_mag_25, 'y': y_25},
    {'name': 'D_AccGyro_13Hz', 'sensors': ['acc', 'gyro'], 'freq': 13, 'X_acc': X_acc_13, 'X_gyro': X_gyro_13, 'X_mag': X_mag_13, 'y': y_13},
    {'name': 'E_AccGyroMag_25Hz', 'sensors': ['acc', 'gyro', 'mag'], 'freq': 25, 'X_acc': X_acc_25, 'X_gyro': X_gyro_25, 'X_mag': X_mag_25, 'y': y_25},
    {'name': 'F_AccGyroMag_13Hz', 'sensors': ['acc', 'gyro', 'mag'], 'freq': 13, 'X_acc': X_acc_13, 'X_gyro': X_gyro_13, 'X_mag': X_mag_13, 'y': y_13},
]

# Dizionario per salvare risultati
results = {}

In [ ]:
# Training loop per tutte le configurazioni
EPOCHS = 50
BATCH_SIZE = 64

for config in configs:
    print("\n" + "="*80)
    print(f"CONFIGURAZIONE: {config['name']}")
    print(f"Sensori: {config['sensors']}, Frequenza: {config['freq']}Hz")
    print("="*80)
    
    # Prepara dati
    X = prepare_data_for_config(config['X_acc'], config['X_gyro'], config['X_mag'], config['y'], config['sensors'])
    y = config['y']
    
    print(f"Input shape: {X.shape}")
    
    # Split train/test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Normalizza
    X_train_norm, X_test_norm, mean, std = normalize_data(X_train, X_test)
    
    # One-hot encoding
    y_train_cat = to_categorical(y_train, NUM_CLASSES)
    y_test_cat = to_categorical(y_test, NUM_CLASSES)
    
    # Crea modello
    input_shape = (X_train_norm.shape[1], X_train_norm.shape[2])
    model = create_cnn_model(input_shape, NUM_CLASSES)
    
    print(f"\nModello creato:")
    model.summary()
    
    # Callbacks
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    
    # Training
    print(f"\nInizio training...")
    start_time = time.time()
    
    history = model.fit(
        X_train_norm, y_train_cat,
        validation_split=0.15,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    
    training_time = time.time() - start_time
    
    # Valutazione
    print(f"\nValutazione su test set...")
    start_time = time.time()
    y_pred = model.predict(X_test_norm)
    inference_time = (time.time() - start_time) / len(X_test_norm) * 1000  # ms per sample
    
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    # Metriche
    accuracy = accuracy_score(y_test, y_pred_classes)
    f1 = f1_score(y_test, y_pred_classes, average='weighted')
    
    print(f"\nRisultati:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-Score (weighted): {f1:.4f}")
    print(f"  Training time: {training_time:.2f}s")
    print(f"  Inference time: {inference_time:.4f}ms/sample")
    
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=ACTIVITIES))
    
    # Salva risultati
    results[config['name']] = {
        'model': model,
        'history': history.history,
        'accuracy': accuracy,
        'f1_score': f1,
        'training_time': training_time,
        'inference_time': inference_time,
        'confusion_matrix': confusion_matrix(y_test, y_pred_classes),
        'classification_report': classification_report(y_test, y_pred_classes, target_names=ACTIVITIES, output_dict=True),
        'input_shape': input_shape,
        'mean': mean,
        'std': std
    }
    
    print(f"\n✓ Configurazione {config['name']} completata!")

---
## 7. Visualizzazione Training History

In [ ]:
# Plot training history per tutte le configurazioni
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Training History - Tutte le Configurazioni', fontsize=16)

for idx, (name, result) in enumerate(results.items()):
    row = idx // 3
    col = idx % 3
    
    ax = axes[row, col]
    history = result['history']
    
    ax.plot(history['accuracy'], label='Train Accuracy', linewidth=2)
    ax.plot(history['val_accuracy'], label='Val Accuracy', linewidth=2)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Confusion Matrix per Tutte le Configurazioni

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Confusion Matrices - Tutte le Configurazioni', fontsize=16)

for idx, (name, result) in enumerate(results.items()):
    row = idx // 3
    col = idx % 3
    
    ax = axes[row, col]
    cm = result['confusion_matrix']
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=[a[:3] for a in ACTIVITIES],
                yticklabels=[a[:3] for a in ACTIVITIES])
    ax.set_title(f"{name}\nAcc: {result['accuracy']:.3f}", fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.show()

---
## 9. Conversione a TensorFlow Lite

In [ ]:
def convert_to_tflite(model, model_name, quantize=False):
    """
    Converte modello Keras a TFLite.
    
    Parameters:
    - model: modello Keras
    - model_name: nome file output
    - quantize: se True, applica quantizzazione int8
    
    Returns:
    - tflite_model_size: dimensione in bytes
    """
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    if quantize:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        model_name = model_name.replace('.tflite', '_quantized.tflite')
    
    tflite_model = converter.convert()
    
    # Salva
    output_path = f'../models/{model_name}'
    with open(output_path, 'wb') as f:
        f.write(tflite_model)
    
    size_kb = len(tflite_model) / 1024
    print(f"✓ Modello salvato: {output_path} ({size_kb:.2f} KB)")
    
    return len(tflite_model)

In [ ]:
# Converti tutti i modelli a TFLite
print("=" * 80)
print("CONVERSIONE A TENSORFLOW LITE")
print("=" * 80)

for name, result in results.items():
    print(f"\nConversione {name}...")
    
    # Versione float32
    size_float = convert_to_tflite(result['model'], f'{name}.tflite', quantize=False)
    
    # Versione quantizzata
    size_quant = convert_to_tflite(result['model'], f'{name}.tflite', quantize=True)
    
    results[name]['tflite_size_float'] = size_float
    results[name]['tflite_size_quant'] = size_quant
    
    print(f"  Float32: {size_float/1024:.2f} KB")
    print(f"  Quantized: {size_quant/1024:.2f} KB")
    print(f"  Riduzione: {(1 - size_quant/size_float)*100:.1f}%")

---
## 10. Test TFLite Inference Performance

In [ ]:
def test_tflite_inference(tflite_path, X_test, n_runs=100):
    """
    Testa performance inferenza TFLite.
    
    Returns:
    - avg_time_ms: tempo medio inferenza in ms
    """
    # Load TFLite model
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Test su n_runs campioni
    times = []
    for i in range(min(n_runs, len(X_test))):
        input_data = X_test[i:i+1].astype(np.float32)
        
        start = time.time()
        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index'])
        times.append((time.time() - start) * 1000)  # ms
    
    avg_time = np.mean(times)
    std_time = np.std(times)
    
    return avg_time, std_time

In [ ]:
# Test inferenza TFLite per tutti i modelli
print("=" * 80)
print("TEST INFERENZA TFLITE")
print("=" * 80)

for config in configs:
    name = config['name']
    
    # Prepara test data
    X = prepare_data_for_config(config['X_acc'], config['X_gyro'], config['X_mag'], config['y'], config['sensors'])
    _, X_test, _, _ = train_test_split(X, config['y'], test_size=0.2, random_state=42)
    X_test_norm, _, _, _ = normalize_data(X, X_test)
    
    # Test float32
    tflite_path_float = f'../models/{name}.tflite'
    if os.path.exists(tflite_path_float):
        avg_time_float, std_time_float = test_tflite_inference(tflite_path_float, X_test_norm)
        results[name]['tflite_inference_time_float'] = avg_time_float
        print(f"\n{name} (Float32):")
        print(f"  Avg inference time: {avg_time_float:.4f} ± {std_time_float:.4f} ms")
    
    # Test quantized
    tflite_path_quant = f'../models/{name}_quantized.tflite'
    if os.path.exists(tflite_path_quant):
        avg_time_quant, std_time_quant = test_tflite_inference(tflite_path_quant, X_test_norm)
        results[name]['tflite_inference_time_quant'] = avg_time_quant
        print(f"  Avg inference time (Quantized): {avg_time_quant:.4f} ± {std_time_quant:.4f} ms")
        print(f"  Speedup: {avg_time_float/avg_time_quant:.2f}x")

---
## 11. Analisi Comparativa - Tabella Riassuntiva

In [ ]:
# Crea tabella comparativa
comparison_data = []

for name, result in results.items():
    comparison_data.append({
        'Configurazione': name,
        'Accuracy': f"{result['accuracy']:.4f}",
        'F1-Score': f"{result['f1_score']:.4f}",
        'Training Time (s)': f"{result['training_time']:.2f}",
        'Keras Inference (ms)': f"{result['inference_time']:.4f}",
        'TFLite Float (ms)': f"{result.get('tflite_inference_time_float', 0):.4f}",
        'TFLite Quant (ms)': f"{result.get('tflite_inference_time_quant', 0):.4f}",
        'Model Size Float (KB)': f"{result['tflite_size_float']/1024:.2f}",
        'Model Size Quant (KB)': f"{result['tflite_size_quant']/1024:.2f}",
    })

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "=" * 120)
print("TABELLA COMPARATIVA - TUTTE LE CONFIGURAZIONI")
print("=" * 120)
print(df_comparison.to_string(index=False))

# Salva CSV
df_comparison.to_csv('../results/model_comparison.csv', index=False)
print("\n✓ Tabella salvata in: ../results/model_comparison.csv")

---
## 12. Grafici Comparativi

In [ ]:
# Estrai dati per i grafici
config_names = list(results.keys())
accuracies = [results[name]['accuracy'] for name in config_names]
f1_scores = [results[name]['f1_score'] for name in config_names]
training_times = [results[name]['training_time'] for name in config_names]
inference_times_keras = [results[name]['inference_time'] for name in config_names]
inference_times_tflite = [results[name].get('tflite_inference_time_quant', 0) for name in config_names]
model_sizes = [results[name]['tflite_size_quant']/1024 for name in config_names]

# Colori per i grafici
colors_25hz = ['#2ecc71', '#27ae60', '#16a085']
colors_13hz = ['#e74c3c', '#c0392b', '#d35400']
colors = colors_25hz + colors_13hz

In [ ]:
# Grafico 1: Accuracy vs Configurazione
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(config_names)), accuracies, color=colors, alpha=0.8, edgecolor='black')
ax.set_xlabel('Configurazione', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Confronto Accuracy tra Configurazioni', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(config_names)))
ax.set_xticklabels(config_names, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1.1])

# Aggiungi valori sopra le barre
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.02, f'{acc:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/accuracy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Grafico 2: Tempo Inferenza TFLite vs Configurazione
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(config_names)), inference_times_tflite, color=colors, alpha=0.8, edgecolor='black')
ax.set_xlabel('Configurazione', fontsize=12, fontweight='bold')
ax.set_ylabel('Tempo Inferenza (ms)', fontsize=12, fontweight='bold')
ax.set_title('Confronto Tempo Inferenza TFLite (Quantized)', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(config_names)))
ax.set_xticklabels(config_names, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

# Aggiungi valori sopra le barre
for bar, time_val in zip(bars, inference_times_tflite):
    ax.text(bar.get_x() + bar.get_width()/2, time_val + 0.01, f'{time_val:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/inference_time_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Grafico 3: Accuracy vs Inference Time (scatter)
fig, ax = plt.subplots(figsize=(10, 8))

for i, name in enumerate(config_names):
    ax.scatter(inference_times_tflite[i], accuracies[i], 
              s=model_sizes[i]*2, alpha=0.6, color=colors[i], edgecolors='black', linewidth=2)
    ax.annotate(name, (inference_times_tflite[i], accuracies[i]), 
               fontsize=9, ha='left', va='bottom')

ax.set_xlabel('Tempo Inferenza TFLite (ms)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Trade-off: Accuracy vs Velocità\n(Dimensione bolla = Dimensione modello)', 
            fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/accuracy_vs_inference_time.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Grafico 4: Effetto rimozione sensori (25Hz vs 13Hz)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 25Hz
acc_25hz = [results['A_Acc_25Hz']['accuracy'], 
            results['C_AccGyro_25Hz']['accuracy'], 
            results['E_AccGyroMag_25Hz']['accuracy']]
labels = ['Acc', 'Acc+Gyro', 'Acc+Gyro+Mag']

ax1.plot(labels, acc_25hz, marker='o', linewidth=3, markersize=10, color='#2ecc71')
ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Effetto Rimozione Sensori @ 25Hz', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1.1])

for i, acc in enumerate(acc_25hz):
    ax1.text(i, acc + 0.02, f'{acc:.3f}', ha='center', fontsize=10, fontweight='bold')

# 13Hz
acc_13hz = [results['B_Acc_13Hz']['accuracy'], 
            results['D_AccGyro_13Hz']['accuracy'], 
            results['F_AccGyroMag_13Hz']['accuracy']]

ax2.plot(labels, acc_13hz, marker='s', linewidth=3, markersize=10, color='#e74c3c')
ax2.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax2.set_title('Effetto Rimozione Sensori @ 13Hz', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1.1])

for i, acc in enumerate(acc_13hz):
    ax2.text(i, acc + 0.02, f'{acc:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/sensor_removal_effect.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Grafico 5: Effetto frequenza (Acc vs Acc+Gyro vs Acc+Gyro+Mag)
fig, ax = plt.subplots(figsize=(10, 6))

x = np.array([0, 1, 2])
width = 0.35

bars1 = ax.bar(x - width/2, acc_25hz, width, label='25Hz', color='#2ecc71', alpha=0.8, edgecolor='black')
bars2 = ax.bar(x + width/2, acc_13hz, width, label='13Hz', color='#e74c3c', alpha=0.8, edgecolor='black')

ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Effetto Frequenza Campionamento', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1.1])

# Aggiungi valori
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/frequency_effect.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 13. Conclusioni e Raccomandazioni

In [ ]:
print("=" * 80)
print("ANALISI E RACCOMANDAZIONI")
print("=" * 80)

# Trova best model per accuracy
best_acc_name = max(results, key=lambda x: results[x]['accuracy'])
best_acc = results[best_acc_name]['accuracy']

# Trova fastest model
fastest_name = min(results, key=lambda x: results[x].get('tflite_inference_time_quant', float('inf')))
fastest_time = results[fastest_name].get('tflite_inference_time_quant', 0)

# Trova smallest model
smallest_name = min(results, key=lambda x: results[x]['tflite_size_quant'])
smallest_size = results[smallest_name]['tflite_size_quant'] / 1024

print(f"\n📊 MIGLIORE ACCURACY:")
print(f"  Configurazione: {best_acc_name}")
print(f"  Accuracy: {best_acc:.4f}")
print(f"  F1-Score: {results[best_acc_name]['f1_score']:.4f}")

print(f"\n⚡ PIÙ VELOCE (TFLite Quantized):")
print(f"  Configurazione: {fastest_name}")
print(f"  Tempo inferenza: {fastest_time:.4f} ms")
print(f"  Accuracy: {results[fastest_name]['accuracy']:.4f}")

print(f"\n💾 PIÙ COMPATTO:")
print(f"  Configurazione: {smallest_name}")
print(f"  Dimensione: {smallest_size:.2f} KB")
print(f"  Accuracy: {results[smallest_name]['accuracy']:.4f}")

print(f"\n\n🎯 RACCOMANDAZIONI:")
print(f"\n1. Per MASSIMA PRECISIONE:")
print(f"   → Usa {best_acc_name}")
print(f"   → Trade-off: maggior tempo di inferenza e consumo batteria")

print(f"\n2. Per DISPOSITIVI MOBILI (balance):")
print(f"   → Usa C_AccGyro_25Hz o D_AccGyro_13Hz")
print(f"   → Buon compromesso tra accuracy e velocità")

print(f"\n3. Per DISPOSITIVI LOW-POWER:")
print(f"   → Usa {smallest_name}")
print(f"   → Minimo consumo energetico")

print(f"\n4. EFFETTO RIMOZIONE SENSORI:")
diff_gyro = results['C_AccGyro_25Hz']['accuracy'] - results['A_Acc_25Hz']['accuracy']
diff_mag = results['E_AccGyroMag_25Hz']['accuracy'] - results['C_AccGyro_25Hz']['accuracy']
print(f"   → Aggiungere Gyro migliora accuracy di: {diff_gyro*100:.2f}%")
print(f"   → Aggiungere Magnetometro migliora accuracy di: {diff_mag*100:.2f}%")

print(f"\n5. EFFETTO FREQUENZA:")
freq_diff_acc = results['A_Acc_25Hz']['accuracy'] - results['B_Acc_13Hz']['accuracy']
print(f"   → Ridurre da 25Hz a 13Hz riduce accuracy di: {freq_diff_acc*100:.2f}%")
time_saved = results['A_Acc_25Hz'].get('tflite_inference_time_quant', 0) - results['B_Acc_13Hz'].get('tflite_inference_time_quant', 0)
print(f"   → Ma riduce tempo inferenza di: {time_saved:.4f} ms ({time_saved/results['A_Acc_25Hz'].get('tflite_inference_time_quant', 1)*100:.1f}%)")

print("\n" + "=" * 80)

---
## 14. Esportazione Risultati Finali

In [ ]:
# Salva tutti i risultati in un file pickle per analisi future
import pickle

# Rimuovi i modelli Keras per ridurre dimensione file
results_export = {}
for name, result in results.items():
    results_export[name] = {k: v for k, v in result.items() if k != 'model'}

with open('../results/all_results.pkl', 'wb') as f:
    pickle.dump(results_export, f)

print("✓ Risultati salvati in: ../results/all_results.pkl")

# Salva anche configurazione dataset
dataset_info = {
    'activities': ACTIVITIES,
    'num_classes': NUM_CLASSES,
    'freq_high': FREQ_HIGH,
    'freq_low': FREQ_LOW,
    'window_size_high': WINDOW_SIZE_HIGH,
    'window_size_low': WINDOW_SIZE_LOW,
    'overlap': OVERLAP,
    'samples_per_activity': SAMPLES_PER_ACTIVITY
}

with open('../results/dataset_info.pkl', 'wb') as f:
    pickle.dump(dataset_info, f)

print("✓ Info dataset salvate in: ../results/dataset_info.pkl")

print("\n" + "=" * 80)
print("✅ TRAINING COMPLETATO CON SUCCESSO!")
print("=" * 80)
print(f"\nModelli TFLite generati: {len(results) * 2} (float + quantized)")
print(f"Risultati salvati in: ../results/")
print(f"Grafici salvati in: ../results/")
print("\nProssimi passi:")
print("  1. Testare modelli TFLite su smartphone Android")
print("  2. Confrontare performance computer vs smartphone")
print("  3. Ottimizzare modello migliore per deployment")
print("=" * 80)